# 2. Embeddings con sentence-transformers

In [1]:
from sentence_transformers import SentenceTransformer
import torch
import re
import numpy as np
import pandas as pd

In [2]:
# Definimos que use el GPU para esta monda
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando: {device}")

Usando: cuda


In [3]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)
print("Modelo cargado. Dimensión de salida:", model.get_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo cargado. Dimensión de salida: 384


Cargamos los datasets

In [4]:
conversaciones = pd.read_parquet("../data/dataset_conversaciones/dataset_50k_anonymized.parquet")

In [5]:
conv_full = (conversaciones
    .sort_values(["conv_id", "date"])
    .groupby("conv_id")
    .agg({
        "user_id": "first",
        "input": lambda x: " ".join(x.astype(str)),
        "channel_source": "first"
    })
    .reset_index()
    .rename(columns={"input": "texto_usuario"}))

print(f"Conversaciones reconstruidas: {len(conv_full):,}")
conv_full.head(2)

Conversaciones reconstruidas: 24,119


,conv_id,user_id,texto_usuario,channel_source
0,0000ACA6-AA00-4227-BCE7-5E78C2742D88,USR-12729,Cuál es su tasa de interés para el crédito de ...,1
1,00043647-377f-4528-b2b3-425afd81f6bd,USR-11649,No le deja seleccionar el cobro para hacer la ...,1


In [6]:
def limpiar(t):
    t = str(t).lower()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"http\S+", "", t)
    return t.strip()
    
conv_full["texto_limpio"] = conv_full["texto_usuario"].apply(limpiar)

# Filtrar conversaciones muy cortas (ruido)
conv_full = conv_full[conv_full["texto_limpio"].str.len() > 10].reset_index(drop=True)
print(f"Conversaciones tras filtrar: {len(conv_full):,}")

Conversaciones tras filtrar: 23,791


In [7]:
textos = conv_full["texto_limpio"].tolist()

embeddings = model.encode(
    textos,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Shape embeddings: {embeddings.shape}")
# Esperado: (n_conversaciones, 384)

Batches:   0%|          | 0/372 [00:00<?, ?it/s]

Shape embeddings: (23791, 384)


In [8]:
np.save("embeddings_conv.npy", embeddings)
conv_full[["conv_id", "user_id"]].to_parquet("conv_index.parquet", index=False)
print("Guardado")

Guardado



# 3. Clustering conversacional con BERTopic

In [10]:
print("Embeddings shape:", embeddings.shape)
print("Conv full:", conv_full.shape)
print("Columnas conv_full:", conv_full.columns.tolist())

Embeddings shape: (23791, 384)
Conv full: (23791, 5)
Columnas conv_full: ['conv_id', 'user_id', 'texto_usuario', 'channel_source', 'texto_limpio']


Cargamos las librerias necesarias

In [16]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

Configurar componentes de BERTopic

In [13]:
# UMAP: reduce dimensiones antes de clusterizar
umap_model = UMAP(
    n_neighbors=15,
    n_components=10,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# HDBSCAN: clustering por densidad
hdbscan_model = HDBSCAN(
    min_cluster_size=80,        # mínimo de convs para formar cluster
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

print("Modelos configurados")


Modelos configurados


In [17]:
from sklearn.feature_extraction.text import CountVectorizer

# Stopwords en español + algunas específicas del dominio
stopwords_es = [
    "de", "la", "que", "el", "en", "y", "a", "los", "del", "se", "las",
    "por", "un", "para", "con", "no", "una", "su", "al", "lo", "como",
    "más", "pero", "sus", "le", "ya", "o", "este", "sí", "porque",
    "esta", "entre", "cuando", "muy", "sin", "sobre", "también", "me",
    "hasta", "hay", "donde", "quien", "desde", "todo", "nos", "durante",
    "todos", "uno", "les", "ni", "contra", "otros", "ese", "eso", "ante",
    "ellos", "e", "esto", "mí", "antes", "algunos", "qué", "unos", "yo",
    "otro", "otras", "otra", "él", "tanto", "esa", "estos", "mucho",
    "quienes", "nada", "muchos", "cual", "poco", "ella", "estar", "estas",
    "algunas", "algo", "nosotros", "mi", "mis", "tú", "te", "ti", "tu",
    "tus", "ellas", "nosotras", "vosotros", "vosotras", "os", "mío",
    "mía", "míos", "mías", "tuyo", "tuya", "tuyos", "tuyas", "suyo",
    "suya", "suyos", "suyas", "nuestro", "nuestra", "nuestros", "nuestras",
    "vuestro", "vuestra", "vuestros", "vuestras", "esos", "esas",
    "hola", "gracias", "buenos", "días", "buenas", "tardes", "noches"
]

vectorizer_model = CountVectorizer(stop_words=stopwords_es, ngram_range=(1, 2))
print("Vectorizer listo")

Vectorizer listo


In [18]:
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

topics, _ = topic_model.fit_transform(
    documents=conv_full["texto_limpio"].tolist(),
    embeddings=embeddings
)

print(f"\nTópicos detectados: {len(set(topics)) - 1}")  # -1 excluye outliers
print(f"Conversaciones en outliers: {(np.array(topics) == -1).sum():,}")

2026-04-25 21:45:37,264 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-25 21:47:18,650 - BERTopic - Dimensionality - Completed ✓
2026-04-25 21:47:18,655 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-25 21:47:25,469 - BERTopic - Cluster - Completed ✓
2026-04-25 21:47:25,490 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-25 21:47:26,937 - BERTopic - Representation - Completed ✓



Tópicos detectados: 70
Conversaciones en outliers: 7,300


### Tópicos detectados

In [19]:
info = topic_model.get_topic_info()
print(f"Total tópicos: {len(info)}")
info.head(20)

Total tópicos: 71


,Topic,Count,Name,Representation,Representative_Docs
0,-1,7300,-1_si_cuenta_tarjeta_crédito,"[si, cuenta, tarjeta, crédito, puedo, es, teng...","[cancelar tarjeta de crédito, cancelar tarjeta..."
1,0,1492,0_tarjeta_tarjeta física_física_nueva,"[tarjeta, tarjeta física, física, nueva, repos...","[venció mi tarjeta física, solicitar tarjeta f..."
2,1,726,1_aclaración_estado_estado cuenta_apartados,"[aclaración, estado, estado cuenta, apartados,...","[aclaración., donde esta mi estado de cuenta, ..."
3,2,609,2_banco_hey banco_hey_banregio,"[banco, hey banco, hey, banregio, clabe, banco...","[número de hey banco, ya no es banregio hey ba..."
4,3,572,3_enganche_000_auto_meses,"[enganche, 000, auto, meses, crédito auto, 60,...",[crédito auto total 190 000 enganche 40 000 a ...
5,4,565,4_tarjeta crédito_crédito_tarjeta_efectivo,"[tarjeta crédito, crédito, tarjeta, efectivo, ...","[efectivo de mi tarjeta de crédito, puedo reti..."
6,5,443,5_cancelar tarjeta_cancelar_tarjeta crédito_qu...,"[cancelar tarjeta, cancelar, tarjeta crédito, ...","[cancelar mi tarjeta de crédito, hola para can..."
7,6,388,6_app_token_aplicación_celular,"[app, token, aplicación, celular, entrar, móvi...","[token móvil, quiero hacer una transferencia y..."
8,7,381,7_apple_apple pay_pay_tarjeta apple,"[apple, apple pay, pay, tarjeta apple, wallet,...","[como verificar la tarjeta para apple pay, com..."
9,8,376,8_reembolso_dinero_compra_devolución,"[reembolso, dinero, compra, devolución, reflej...",[tengo u. reembolso pendiente y aún no se me v...


In [20]:
for topic_id in info["Topic"].head(10).tolist():
    if topic_id == -1:
        continue
    print(f"\n{'='*60}")
    print(f"TÓPICO {topic_id} · {info[info.Topic==topic_id]['Count'].values[0]} convs")
    print(f"Palabras clave: {[w for w, _ in topic_model.get_topic(topic_id)[:10]]}")
    print(f"\nEjemplo de conversación:")
    ejemplos = [conv_full.iloc[i]['texto_limpio'][:300] 
                for i in range(len(topics)) if topics[i] == topic_id][:2]
    for e in ejemplos:
        print(f"  → {e}")


TÓPICO 0 · 1492 convs
Palabras clave: ['tarjeta', 'tarjeta física', 'física', 'nueva', 'reposición', 'venció', 'solicitar', 'nueva tarjeta', 'costo', 'tarjeta fisica']

Ejemplo de conversación:
  → hola no se si mi tarjeta está prendida
  → hola solo para ver eficaz si mi tarjeta ya quedó activada ok gracias no 👍

TÓPICO 1 · 726 convs
Palabras clave: ['aclaración', 'estado', 'estado cuenta', 'apartados', 'cobro', 'apartado', 'estatus', 'estatus aclaración', 'cajeros', 'folio']

Ejemplo de conversación:
  → necesito meter una aclaración
  → porque al depositarme aparace que no existe

TÓPICO 2 · 609 convs
Palabras clave: ['banco', 'hey banco', 'hey', 'banregio', 'clabe', 'bancos', 'clave', 'independiente', 'cuenta', 'banco es']

Ejemplo de conversación:
  → número hey banco
  → número de hey banco

TÓPICO 3 · 572 convs
Palabras clave: ['enganche', '000', 'auto', 'meses', 'crédito auto', '60', '60 meses', 'mil', 'crédito', '48']

Ejemplo de conversación:
  → crédito auto si 100000 a 24 

In [22]:
fig = topic_model.visualize_topics()
fig.show()

In [24]:
fig = topic_model.visualize_barchart(top_n_topics=12, n_words=8)
fig.show()

In [23]:
conv_full["topic"] = topics
conv_full["topic_name"] = conv_full["topic"].map(
    dict(zip(info["Topic"], info["Name"]))
)

print(conv_full[["conv_id", "user_id", "topic", "topic_name"]].head(10))

                                conv_id    user_id  topic  \
0  0000ACA6-AA00-4227-BCE7-5E78C2742D88  USR-12729     -1   
1  00043647-377f-4528-b2b3-425afd81f6bd  USR-11649     -1   
2  000502c2-288c-41f6-b751-a8b45a376a81  USR-09092     -1   
3  000DD932-B649-41FF-9B2A-EB08B33F5072  USR-10602     -1   
4  00149741-F4D6-4BC9-B8FB-6FFD398FD73F  USR-01560      9   
5  0014FB3D-727F-43B8-9E9C-1F63C0303818  USR-06635     -1   
6  001ea1e9-93aa-4d59-b19f-eb11bb3aa312  USR-00940     11   
7  00221607-cdb2-47f2-a9ac-913d4c7744c6  USR-09344     38   
8  00243113-211B-4123-8C2F-2AEDC69D452E  USR-09850      1   
9  002D9FA0-5064-407C-B4CE-480825090DD5  USR-10380      1   

                                          topic_name  
0                       -1_si_cuenta_tarjeta_crédito  
1                       -1_si_cuenta_tarjeta_crédito  
2                       -1_si_cuenta_tarjeta_crédito  
3                       -1_si_cuenta_tarjeta_crédito  
4      9_cliente_atención_clientes_atención clientes 

In [26]:
# Después de ver los tópicos, decides cuántos perfiles macro quieres
# Por ejemplo: agrupar 30 tópicos en 8 perfiles

# Opción automática: que BERTopic los reduzca por similitud
topic_model.reduce_topics(
    docs=conv_full["texto_limpio"].tolist(),
    nr_topics=10  # número objetivo de perfiles
)

# Re-asignar tópicos
conv_full["perfil_conv"] = topic_model.topics_
info_reducido = topic_model.get_topic_info()
print(info_reducido)

2026-04-25 22:21:59,919 - BERTopic - Topic reduction - Reducing number of topics
2026-04-25 22:21:59,920 - BERTopic - Topic reduction - Number of topics (10) is equal or higher than the clustered topics(8).
2026-04-25 22:21:59,922 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-25 22:22:01,030 - BERTopic - Representation - Completed ✓


   Topic  Count                                             Name  \
0     -1   7300                       -1_tarjeta_si_cuenta_puedo   
1      0  11944                       0_tarjeta_crédito_puedo_si   
2      1   1759               1_cargo_estado_clabe_estado cuenta   
3      2   1350                        2_app_apple_pay_apple pay   
4      3   1093                 3_atención_número_cliente_hablar   
5      4    155                4_codi_pagar codi_codi cómo_pagar   
6      5    108  5_domicilio_dirección_cambiar_cambiar domicilio   
7      6     82        6_plástico_nuevo_plástico tarjeta_tarjeta   

                                      Representation  \
0  [tarjeta, si, cuenta, puedo, crédito, es, teng...   
1  [tarjeta, crédito, puedo, si, es, quiero, cuen...   
2  [cargo, estado, clabe, estado cuenta, puedo, a...   
3  [app, apple, pay, apple pay, tarjeta, aplicaci...   
4  [atención, número, cliente, hablar, asesor, cl...   
5  [codi, pagar codi, codi cómo, pagar, pago codi..

In [27]:
for pid in sorted(set(conv_full["perfil_conv"])):
    if pid == -1:
        continue
    palabras = [w for w, _ in topic_model.get_topic(pid)[:8]]
    n = (conv_full["perfil_conv"] == pid).sum()
    print(f"\nPerfil {pid} · {n} convs")
    print(f"  Palabras: {', '.join(palabras)}")



Perfil 0 · 11944 convs
  Palabras: tarjeta, crédito, puedo, si, es, quiero, cuenta, cómo

Perfil 1 · 1759 convs
  Palabras: cargo, estado, clabe, estado cuenta, puedo, aclaración, reconocido, nip

Perfil 2 · 1350 convs
  Palabras: app, apple, pay, apple pay, tarjeta, aplicación, puedo, cómo

Perfil 3 · 1093 convs
  Palabras: atención, número, cliente, hablar, asesor, clientes, teléfono, atención clientes

Perfil 4 · 155 convs
  Palabras: codi, pagar codi, codi cómo, pagar, pago codi, qr, codi codi, código

Perfil 5 · 108 convs
  Palabras: domicilio, dirección, cambiar, cambiar domicilio, cambio domicilio, cambio, domicilio cómo, cambiar dirección

Perfil 6 · 82 convs
  Palabras: plástico, nuevo, plástico tarjeta, tarjeta, plastico, plástico nuevo, costo, si
